# Bifurcation + Lyapunov animation
Run the cell and press Play. Both plots grow together as r increases. Use the slider to inspect r ≈ 3.2, 3.5, 3.7, and 3.83.
Notice that a stable cycle can have a negative Lyapunov exponent, and that periodic windows interrupt the region of positive exponents.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass


def logistic_step(x, r):
    """One update; use x in [0, 1] and r in [0, 4]."""
    return r * x * (1 - x)


def prepare_diagnostics(r_values, x0=0.1, burn_in=12000,
                        n_average=4000, n_keep=192):
    """Finite-time Lyapunov estimates and late values from the same runs.

    Average log|f'(x_t)| before advancing x_t. A zero derivative gives
    -infinity; no artificial epsilon is added to the derivative.
    """
    r_values = np.asarray(r_values, dtype=float)
    if r_values.ndim != 1 or not np.all(np.isfinite(r_values)):
        raise ValueError("r_values must be a finite one-dimensional array.")
    if np.any((r_values < 0) | (r_values > 4)) or not 0 <= x0 <= 1:
        raise ValueError("Use r in [0, 4] and x0 in [0, 1].")
    if not 1 <= n_keep <= n_average or burn_in < 0:
        raise ValueError("Require 1 <= n_keep <= n_average and burn_in >= 0.")
    x = np.full(r_values.size, float(x0))
    for _ in range(burn_in):
        x = logistic_step(x, r_values)
    total = np.zeros_like(x)
    late = np.empty((r_values.size, n_keep))
    for t in range(n_average):
        with np.errstate(divide="ignore"):
            total += np.log(np.abs(r_values * (1 - 2*x)))
        x = logistic_step(x, r_values)
        if t >= n_average - n_keep:
            late[:, t - (n_average - n_keep)] = x
    return late, total / n_average


def show_lyapunov_animation():
    r_values = np.linspace(2.5, 4.0, 1501)
    late, lyapunov = prepare_diagnostics(r_values)
    bif_r = np.repeat(r_values, late.shape[1])
    bif_x = late.ravel()
    floor, ceiling = -2.0, 0.85
    plotted_lambda = np.clip(lyapunov, floor, ceiling)

    play = widgets.Play(value=0, min=0, max=1500, step=10,
                        interval=100, repeat=False)
    slider = widgets.IntSlider(value=0, min=0, max=1500,
                               description="r position:", readout=False,
                               continuous_update=False,
                               layout=widgets.Layout(width="450px"))
    reset = widgets.Button(description="Reset")
    readout = widgets.HTML()
    output = widgets.Output()
    link = widgets.link((play, "value"), (slider, "value"))

    def draw(change=None):
        i = slider.value
        r, lam = r_values[i], lyapunov[i]
        if abs(lam) < 0.01:
            meaning, color = "Near zero: inspect convergence", "#777777"
        elif lam < 0:
            meaning, color = "Nearby errors shrink on average", "#20804d"
        else:
            meaning, color = "Nearby errors grow on average", "#c74343"
        value = f"{lam:.4f}" if np.isfinite(lam) else "−∞"
        readout.value = (f"<b>r = {r:.3f} &nbsp; λ ≈ {value}</b>"
                         f" &nbsp; <span style='color:{color}'>{meaning}</span>")
        with output:
            clear_output(wait=True)
            fig, (top, bottom) = plt.subplots(2, 1, figsize=(10, 8),
                                              sharex=True, constrained_layout=True)
            end = (i+1)*late.shape[1]
            top.plot(bif_r[:end], bif_x[:end], ",", color="black", alpha=0.45)
            top.scatter(np.full(late.shape[1], r), late[i], s=9,
                        color="tab:orange", alpha=0.75, linewidths=0, zorder=4)
            top.set(ylim=(-0.03, 1.03), ylabel="Late values of x",
                    title="Bifurcation diagram")
            rr, ll = r_values[:i+1], plotted_lambda[:i+1]
            bottom.axhline(0, color="0.35", linestyle="--", lw=1)
            bottom.plot(rr, ll, color="royalblue", lw=1.1)
            bottom.fill_between(rr, ll, 0, where=ll < 0, interpolate=True,
                                color="#70b980", alpha=0.25, label="λ < 0: contraction")
            bottom.fill_between(rr, ll, 0, where=ll > 0, interpolate=True,
                                color="#e57a7a", alpha=0.3, label="λ > 0: expansion")
            bottom.scatter([r], [plotted_lambda[i]], color=color, s=45, zorder=5)
            clipped = lyapunov[:i+1] < floor
            if np.any(clipped):
                bottom.scatter(rr[clipped], np.full(clipped.sum(), floor),
                               marker="v", s=18, color="royalblue", clip_on=False)
            bottom.set(ylim=(floor, ceiling), xlabel="Parameter r",
                       ylabel="Estimated Lyapunov exponent λ",
                       title="Average log stretching along the trajectory")
            bottom.legend(loc="lower left", fontsize=9)
            for ax in (top, bottom):
                ax.set_xlim(2.48, 4.02)
                ax.axvline(r, color="tab:orange", lw=0.9, alpha=0.7)
                ax.grid(alpha=0.15)
            fig.suptitle(f"Same parameter, two views | r = {r:.3f}", fontweight="bold")
            plt.show()
            plt.close(fig)

    def restart(button):
        play.playing = False
        slider.value = 0

    slider.observe(draw, names="value")
    reset.on_click(restart)
    display(widgets.VBox([
        widgets.HTML("<b>One Play button reveals both plots together.</b>"),
        widgets.HBox([play, reset]), slider, readout,
        widgets.HTML("λ ≈ mean(log |r(1 − 2xₜ)|). Each run starts at x₀ = 0.1; "
                     "discard 12,000 updates, then average 4,000 local log slopes. "
                     "The bifurcation panel shows the last 192 values."),
        output,
        widgets.HTML("Negative λ occurs for stable fixed points <i>and cycles</i>. "
                     "Positive λ indicates average local expansion along this trajectory. "
                     "These are finite-time estimates; near zero needs care. "
                     "▼ marks values below the plotted limit −2; the readout keeps "
                     "the computed value. No epsilon is added to zero slopes."),
    ]))
    draw()
    return link


lyapunov_link = show_lyapunov_animation()
